In [1]:
%pip install -q gspread google-auth PyMuPDF langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving cleaned_apartment_rentals_prepared.csv to cleaned_apartment_rentals_prepared (1).csv


In [ ]:
import pandas as pd

CLEANED_DATASET_PATH = next(iter(uploaded))

listings_prepared = pd.read_csv(CLEANED_DATASET_PATH)

print(f"Rows: {len(listings_prepared):,}")
print(f"Columns: {len(listings_prepared.columns)}")

In [ ]:
# Inspect All Dataset Columns

print(f"Total columns: {len(listings_prepared.columns)}\n")

for number, column in enumerate(listings_prepared.columns, start=1): # printing position and column name
    print(f"{number}. {column}")

# **PDF Dataset**

In [ ]:
#  Download the Housing Guidance PDF

import requests
from pathlib import Path

PDF_URL = (
    "https://www.ifegypte.com/wp-content/uploads/2024/07/"
    "Guide-dinstallation-EN-1.pdf"
)

PDF_PATH = Path("Guide_to_Settling_in_Egypt.pdf")

response = requests.get(PDF_URL, timeout=60)
response.raise_for_status()

# Verify that the downloaded file is actually a PDF.
if not response.content.startswith(b"%PDF"):
    raise ValueError(
        "The downloaded file does not appear to be a PDF. "
        "We will upload the PDF manually instead."
    )

PDF_PATH.write_bytes(response.content)

print("PDF downloaded successfully!")
print(f"File name: {PDF_PATH.name}")
print(f"File size: {PDF_PATH.stat().st_size / 1024:.2f} KB")

In [ ]:
%pip install -q PyMuPDF

In [ ]:
# Extract Text from the PDF

import fitz  # PyMuPDF

pdf_document = fitz.open(str(PDF_PATH))

pdf_pages = []
pages_without_text = []

for page_number, page in enumerate(pdf_document, start=1):
    page_text = page.get_text("text").strip()

    if page_text:
        pdf_pages.append({
            "page_number": page_number,
            "text": page_text
        })
    else:
        pages_without_text.append(page_number)

print("PDF opened successfully!")
print("Total PDF pages:", len(pdf_document))
print("Pages with extractable text:", len(pdf_pages))
print("Pages without extractable text:", len(pages_without_text))

if pages_without_text:
    print("Pages needing further inspection:", pages_without_text)

print("\n=== Text Preview from the First Page with Text ===")

if pdf_pages:
    print("Page:", pdf_pages[0]["page_number"])
    print(pdf_pages[0]["text"][:2000])
else:
    print("No extractable text was found.")

In [ ]:
# Prepare PDF Pages as LangChain Documents

import re
import pymupdf
from langchain_core.documents import Document

pdf_document = pymupdf.open(str(PDF_PATH))

pdf_documents = []
skipped_pages = []

for page_index, page in enumerate(pdf_document):
    # Extract and normalize the page text.
    text = page.get_text("text").strip()

    text = re.sub(r"\n[ \t]*\n(?:[ \t]*\n)+", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = text.strip()

    # Skip blank pages and pages containing only short headings.
    if len(text) < 100:
        skipped_pages.append(page_index + 1)
        continue

    # Store the page as a LangChain Document.
    document = Document(
        page_content=text,
        metadata={
            "source": PDF_PATH.name,
            "page": page_index + 1
        }
    )

    pdf_documents.append(document)

print("=== PDF Document Preparation ===")
print("Total PDF pages:", len(pdf_document))
print("Pages retained:", len(pdf_documents))
print("Pages skipped:", len(skipped_pages))
print("Skipped physical PDF pages:", skipped_pages)

print("\n=== Example Document ===")

if pdf_documents:
    example_doc = next(
        (
            doc for doc in pdf_documents
            if doc.metadata["page"] == 16
        ),
        pdf_documents[0]
    )

    print("Metadata:", example_doc.metadata)
    print("\nExtracted text preview:\n")
    print(example_doc.page_content[:1200])

In [ ]:
%pip install -q langchain-text-splitters

In [ ]:
# Split PDF Documents into Chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    add_start_index=True
)

pdf_chunks = text_splitter.split_documents(pdf_documents)

print("=== Text Chunking Summary ===")
print("Original PDF documents:", len(pdf_documents))
print("Total chunks created:", len(pdf_chunks))

chunk_lengths = [
    len(chunk.page_content)
    for chunk in pdf_chunks
]

print("Shortest chunk:", min(chunk_lengths), "characters")
print("Longest chunk:", max(chunk_lengths), "characters")
print(
    "Average chunk length:",
    round(sum(chunk_lengths) / len(chunk_lengths), 2),
    "characters"
)

print("\n=== Example Chunk ===")

example_chunk = next(
    (
        chunk for chunk in pdf_chunks
        if chunk.metadata.get("page") == 16
    ),
    pdf_chunks[0]
)

print("Metadata:", example_chunk.metadata)
print("\nText:\n")
print(example_chunk.page_content)

In [ ]:
# Find PDF chunks shorter than 100 characters
short_chunks = [
    (i, len(doc.page_content.strip()), doc.metadata, repr(doc.page_content))
    for i, doc in enumerate(pdf_chunks)
    if len(doc.page_content.strip()) < 100
]

print(f"Number of short chunks: {len(short_chunks)}\n")

for index, length, metadata, content in short_chunks:
    print(f"Chunk index: {index}")
    print(f"Length: {length}")
    print(f"Metadata: {metadata}")
    print(f"Content: {content}")
    print("-" * 60)

In [ ]:
# Keep the original chunks unchanged and filter out very short chunks
pdf_chunks_filtered = [
    doc for doc in pdf_chunks
    if len(doc.page_content.strip()) >= 50
]

# Summarize the filtering results
chunk_lengths = [
    len(doc.page_content.strip())
    for doc in pdf_chunks_filtered
]

print("=== Filtered Chunking Summary ===")
print(f"Original chunks: {len(pdf_chunks)}")
print(f"Chunks retained: {len(pdf_chunks_filtered)}")
print(f"Chunks removed: {len(pdf_chunks) - len(pdf_chunks_filtered)}")
print(f"Shortest retained chunk: {min(chunk_lengths)} characters")
print(f"Longest retained chunk: {max(chunk_lengths)} characters")
print(f"Average retained chunk length: {sum(chunk_lengths) / len(chunk_lengths):.2f} characters")

# Confirm that the useful rental tip was preserved
print("\n=== Example of a Preserved Rental Tip ===")
for doc in pdf_chunks_filtered:
    if "Take a picture of all the counters" in doc.page_content:
        print(f"Page: {doc.metadata['page']}")
        print(doc.page_content)
        break

In [ ]:
#convert property listings (excel) into langchain document


from langchain_core.documents import Document
import pandas as pd

def get_text(row, column, fallback_column=None):
    """Return a cleaned text value, handling missing data."""
    value = row.get(column, None)

    if (value is None or pd.isna(value)) and fallback_column:
        value = row.get(fallback_column, None)

    if value is None or pd.isna(value):
        return "Not specified"

    value = str(value).strip()
    return value if value else "Not specified"


def get_number(row, column):
    """Return a numeric value or None if missing or invalid."""
    value = pd.to_numeric(row.get(column), errors="coerce")

    if pd.isna(value):
        return None

    return float(value)


listing_documents = []

for _, row in listings_prepared.iterrows():

    title = get_text(row, "title_clean", "title")
    description = get_text(row, "description_clean", "description")
    amenities = get_text(row, "amenities_clean", "amenities")
    location = get_text(row, "location_full")

    city = get_text(row, "city")
    property_type = get_text(row, "property_type")
    bedrooms = get_text(row, "bedrooms_normalized")
    bathrooms = get_text(row, "bathrooms_numeric")
    area = get_text(row, "area_numeric")
    furnished = get_text(row, "furnished_normalized")

    price = get_number(row, "price_numeric")
    price_period = get_text(row, "price_period")

    # Create searchable text for this rental listing
    page_content = f"""
    Rental Property
    Title: {title}
    Property Type: {property_type}
    City: {city}
    Location: {location}
    Monthly/Periodic Rent: {price if price is not None else "Not specified"} EGP
    Price Period: {price_period}
    Bedrooms: {bedrooms}
    Bathrooms: {bathrooms}
    Area: {area} {get_text(row, "area_unit")}
    Furnished: {furnished}
    Amenities: {amenities}
    Description: {description}
    """

    # Keep structured information as metadata for later retrieval
    metadata = {
        "source_type": "property_listing",
        "listing_id": str(row.get("listing_id", "")),
        "city": city,
        "property_type": property_type,
        "bedrooms": bedrooms,
        "furnished": furnished,
        "price_period": price_period,
        "detail_url": get_text(row, "detail_url"),
    }

    if price is not None:
        metadata["price_egp"] = price

    listing_documents.append(
        Document(
            page_content=page_content.strip(),
            metadata=metadata
        )
    )

print("=== Property Document Summary ===")
print(f"Clean rental listings: {len(listings_prepared):,}")
print(f"Documents created: {len(listing_documents):,}")

print("\n=== Example Property Document ===")
print("Metadata:", listing_documents[0].metadata)
print("\nText:\n", listing_documents[0].page_content[:1200])

In [ ]:
#This cell cleans the URLs so each property Document stores a usable, consistently formatted link
import re
from urllib.parse import urlparse

def normalize_detail_url(value):
    """Extract a clean URL from a plain or Markdown-formatted link."""
    if value is None:
        return ""

    url = str(value).strip()

    # Handle Markdown format: [label](https://example.com)
    match = re.fullmatch(r"\[([^\]]+)\]\((https?://[^)]+)\)", url)

    if match:
        url = match.group(2).strip()

    # Keep only valid HTTP/HTTPS URLs with a hostname
    try:
        parsed = urlparse(url)
        if parsed.scheme in ("http", "https") and parsed.netloc:
            return url
    except ValueError:
        pass

    return ""


# Clean the URL metadata in the property documents
for doc in listing_documents:
    doc.metadata["detail_url"] = normalize_detail_url(
        doc.metadata.get("detail_url", "")
    )

valid_urls = sum(
    bool(doc.metadata["detail_url"])
    for doc in listing_documents
)

print("=== Property URL Validation ===")
print(f"Total property documents: {len(listing_documents):,}")
print(f"Documents with valid URLs: {valid_urls:,}")
print(f"Documents without valid URLs: {len(listing_documents) - valid_urls:,}")

print("\n=== Example Clean URL ===")
print(listing_documents[0].metadata["detail_url"])

# **Data Integration**

In [ ]:
from collections import Counter
#This cell combines both sources into one collection of LangChain Documents

# Label the PDF chunks so we can distinguish them from property listings
for doc in pdf_chunks_filtered:
    doc.metadata["source_type"] = "housing_guide"

# Combine both sources
all_documents = listing_documents + pdf_chunks_filtered

# Display a summary
source_counts = Counter(
    doc.metadata["source_type"] for doc in all_documents
)

print("=== Combined Documents Summary ===")
print(f"Property listing documents: {len(listing_documents):,}")
print(f"PDF guidance chunks: {len(pdf_chunks_filtered):,}")
print(f"Total documents: {len(all_documents):,}")

print("\n=== Documents by Source ===")
for source, count in source_counts.items():
    print(f"{source}: {count:,}")

print("\n=== Example PDF Metadata ===")
print(pdf_chunks_filtered[0].metadata)

# **Embedding**

In [ ]:
%pip install -q langchain-huggingface sentence-transformers

In [ ]:
# testing the embeddeding model
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# Use the GPU if available; otherwise, use the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

# Select one property listing and one housing-guide chunk
guide_document = next(
    doc for doc in all_documents
    if doc.metadata["source_type"] == "housing_guide"
)

test_documents = [
    listing_documents[0],
    guide_document
]

# Generate embeddings for these two documents
test_embeddings = embedding_model.embed_documents(
    [doc.page_content for doc in test_documents]
)

print("\n=== Embedding Test Results ===")
print(f"Documents tested: {len(test_embeddings)}")
print(f"Embedding dimensions: {len(test_embeddings[0])}")
print(f"Property embedding dimensions: {len(test_embeddings[0])}")
print(f"Guide embedding dimensions: {len(test_embeddings[1])}")
print(f"All values are finite: {all(torch.isfinite(torch.tensor(e)).all().item() for e in test_embeddings)}")

Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=== Embedding Test Results ===
Documents tested: 2
Embedding dimensions: 384
Property embedding dimensions: 384
Guide embedding dimensions: 384
All values are finite: True


In [ ]:
%pip install -q langchain-community faiss-cpu

In [ ]:
%pip install -q requests==2.32.4

In [ ]:
import requests
from langchain_community.vectorstores import FAISS
import faiss

print("Requests version:", requests.__version__)
print("LangChain FAISS import: successful")
print("FAISS version:", faiss.__version__)

Requests version: 2.34.2
LangChain FAISS import: successful
FAISS version: 1.15.1


In [ ]:
# Generate all embeddings and build the FAISS index
import time
from langchain_community.vectorstores import FAISS

# Start timing the indexing process
start_time = time.time()

print("Generating embeddings and building the FAISS index...")
print(f"Documents to process: {len(all_documents):,}")

# Embed all documents and store them in FAISS
vectorstore = FAISS.from_documents(
    documents=all_documents,
    embedding=embedding_model
)

# Verify that all documents were indexed
indexed_count = vectorstore.index.ntotal
embedding_dimension = vectorstore.index.d

print("\n=== FAISS Index Summary ===")
print(f"Documents prepared: {len(all_documents):,}")
print(f"Documents indexed: {indexed_count:,}")
print(f"Embedding dimensions: {embedding_dimension}")
print(f"All documents indexed: {indexed_count == len(all_documents)}")
print(f"Time elapsed: {(time.time() - start_time):.2f} seconds")

Generating embeddings and building the FAISS index...
Documents to process: 14,672

=== FAISS Index Summary ===
Documents prepared: 14,672
Documents indexed: 14,672
Embedding dimensions: 384
All documents indexed: True
Time elapsed: 41.08 seconds


# **RAG**

In [ ]:
# Test retrieval with metadata filters
# Testing for each source, not mixing the two sources

test_cases = [
    {
        "name": "Property Search",
        "query": (
            "I want to rent a two-bedroom apartment in New Cairo "
            "with monthly rent around 40000 EGP."
        ),
        "source_type": "property_listing"
    },
    {
        "name": "Housing Guidance",
        "query": (
            "What should I check and agree with the landlord "
            "before renting an apartment in Egypt?"
        ),
        "source_type": "housing_guide"
    }
]

for test in test_cases:
    print(f"\n{'=' * 60}")
    print(f"Test: {test['name']}")
    print("=" * 60)

    results = vectorstore.similarity_search(
        query=test["query"],
        k=3,
        filter={"source_type": test["source_type"]}
    )

    for rank, doc in enumerate(results, start=1):
        print(f"\nResult {rank}")
        print(f"Source: {doc.metadata['source_type']}")

        if test["source_type"] == "property_listing":
            print(f"City: {doc.metadata.get('city')}")
            print(f"Bedrooms: {doc.metadata.get('bedrooms')}")
            print(f"Monthly rent: {doc.metadata.get('price_egp')} EGP")
            print(f"Location: {doc.metadata.get('detail_url', '')}")
        else:
            print(f"PDF page: {doc.metadata.get('page')}")

        print("Text:", doc.page_content[:300].strip())


Test: Property Search

Result 1
Source: property_listing
City: Cairo
Bedrooms: 2
Monthly rent: 85000.0 EGP
Location: https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-the-5th-settlement-5th-settlement-compounds-eastown-2PbhpJ2K824.html
Text: Rental Property
    Title: Fully Furnished Apartment for Rent in Eastown – Ne
    Property Type: Apartment
    City: Cairo
    Location: Eastown, 5th Settlement Compounds, The 5th Settlement, New Cairo City, Cairo
    Monthly/Periodic Rent: 85000.0 EGP
    Price Period: monthly
    Bedrooms: 2
    B

Result 2
Source: property_listing
City: Cairo
Bedrooms: 2
Monthly rent: 75000.0 EGP
Location: https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-north-investors-area-cairo-festival-city-2Pp2BfR1xFg.html
Text: Rental Property
    Title: Fully Furnished Apartment for Rent +Private Garden
    Property Type: Apartment
    City: Cairo
    Location: Cairo Festival City, North Investors Area, New

In [ ]:
# Add structured location metadata to documents stored in FAISS

import pandas as pd

# Create a lookup from each listing ID to its original location fields
listing_lookup = listings_prepared.copy()

listing_lookup["_listing_id_key"] = (
    listing_lookup["listing_id"].astype(str)
)

listing_lookup = (
    listing_lookup
    .drop_duplicates("_listing_id_key")
    .set_index("_listing_id_key")
)

location_fields = [
    "location_full",
    "town",
    "district",
    "subdistrict"
]

updated_count = 0
sample_document = None

# Update the documents already stored in the FAISS docstore
for store_id in vectorstore.index_to_docstore_id.values():
    doc = vectorstore.docstore.search(store_id)

    if doc.metadata.get("source_type") != "property_listing":
        continue

    listing_id = str(doc.metadata.get("listing_id", ""))

    if listing_id not in listing_lookup.index:
        continue

    row = listing_lookup.loc[listing_id]

    for field in location_fields:
        value = row.get(field, "")

        if value is None or pd.isna(value):
            value = ""

        doc.metadata[field] = str(value).strip()

    updated_count += 1

    if sample_document is None:
        sample_document = doc

print("=== Location Metadata Summary ===")
print(f"Property documents updated: {updated_count:,}")

if sample_document:
    for field in [
        "city",
        "location_full",
        "town",
        "district",
        "subdistrict",
        "bedrooms",
        "price_egp"
    ]:
        print(f"{field}: {sample_document.metadata.get(field, 'Not specified')}")

=== Location Metadata Summary ===
Property documents updated: 14,579
city: Cairo
location_full: El Patio Oro, 5th Settlement Compounds, The 5th Settlement, New Cairo City, Cairo
town: New Cairo City
district: The 5th Settlement
subdistrict: 5th Settlement Compounds
bedrooms: 2
price_egp: 35000.0


In [ ]:
# Hybrid retrieval: semantic ranking + strict property filters
#solking criteria and conditions problems in property

query = (
    "I want to rent a two-bedroom apartment in New Cairo "
    "with monthly rent around 40000 EGP."
)

max_budget = 40000
required_bedrooms = "2"
location_query = "new cairo"
top_k = 5

#Retrieve documents in semantic order
print("Searching the FAISS index...")
ranked_results = vectorstore.similarity_search_with_score(
    query,
    k=vectorstore.index.ntotal
)

matching_properties = []
matching_count = 0

for doc, distance in ranked_results:
    metadata = doc.metadata

    # Only consider property listings
    if metadata.get("source_type") != "property_listing":
        continue

    # Enforce the maximum monthly budget
    price = metadata.get("price_egp")
    if price is None or float(price) > max_budget:
        continue

    if metadata.get("price_period", "").lower() != "monthly":
        continue

    # Enforce the bedroom count
    bedrooms = str(metadata.get("bedrooms", "")).strip()
    if bedrooms != required_bedrooms:
        continue

    # Match the requested location against all available location fields
    location_text = " ".join(
        str(metadata.get(field, ""))
        for field in [
            "city", "location_full", "town", "district", "subdistrict"
        ]
    ).casefold()

    if location_query not in location_text:
        continue

    matching_count += 1

    # Keep the highest-ranked matching results
    if len(matching_properties) < top_k:
        matching_properties.append((doc, distance))

print("\n=== Strict Filtering Results ===")
print(f"Maximum monthly budget: {max_budget:,} EGP")
print(f"Required bedrooms: {required_bedrooms}")
print(f"Requested location: New Cairo")
print(f"Properties matching all filters: {matching_count:,}")
print(f"Properties displayed: {len(matching_properties)}")

for rank, (doc, distance) in enumerate(matching_properties, start=1):
    metadata = doc.metadata

    print(f"\n--- Property {rank} ---")
    print(f"Location: {metadata.get('location_full', 'Not specified')}")
    print(f"Monthly rent: {metadata.get('price_egp'):,.0f} EGP")
    print(f"Bedrooms: {metadata.get('bedrooms')}")
    print(f"Furnished: {metadata.get('furnished', 'Not specified')}")
    print(f"URL: {metadata.get('detail_url', '')}")
    print(f"Semantic distance: {distance:.4f}")

Searching the FAISS index...

=== Strict Filtering Results ===
Maximum monthly budget: 40,000 EGP
Required bedrooms: 2
Requested location: New Cairo
Properties matching all filters: 799
Properties displayed: 5

--- Property 1 ---
Location: The Address East, 90 Street, The 5th Settlement, New Cairo City, Cairo
Monthly rent: 20,000 EGP
Bedrooms: 2
Furnished: PARTLY
URL: https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-the-5th-settlement-90-street-the-address-east-8705718.html
Semantic distance: 0.4082

--- Property 2 ---
Location: The Village, South Investors Area, New Cairo City, Cairo
Monthly rent: 33,000 EGP
Bedrooms: 2
Furnished: PARTLY
URL: https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-south-investors-area-the-village-2Ps4J7eplhI.html
Semantic distance: 0.4373

--- Property 3 ---
Location: The Village, South Investors Area, New Cairo City, Cairo
Monthly rent: 35,000 EGP
Bedrooms: 2
Furnished: PARTLY
URL: https://ww

In [ ]:
import pandas as pd

# Add exact amenity metadata to the existing property documents.
# This does not rebuild the FAISS index or regenerate embeddings.

amenity_lookup = listings_prepared.copy()
amenity_lookup["_listing_id_key"] = (
    amenity_lookup["listing_id"].astype(str)
)

amenity_lookup = (
    amenity_lookup
    .drop_duplicates("_listing_id_key")
    .set_index("_listing_id_key")
)

for store_id in vectorstore.index_to_docstore_id.values():
    doc = vectorstore.docstore.search(store_id)

    if doc.metadata.get("source_type") != "property_listing":
        continue

    listing_id = str(doc.metadata.get("listing_id", ""))

    if listing_id not in amenity_lookup.index:
        continue

    row = amenity_lookup.loc[listing_id]
    amenities_value = row.get(
        "amenities_clean",
        row.get("amenities", "")
    )

    if amenities_value is None or pd.isna(amenities_value):
        amenities_value = ""

    doc.metadata["amenities"] = str(amenities_value).strip()


def search_properties(
    query,
    max_budget=None,
    bedrooms=None,
    location=None,
    furnished=None,
    amenities=None,
    top_k=5
):
    """Retrieve semantically relevant apartments matching strict filters."""

    # Search the index in semantic-similarity order.
    ranked_results = vectorstore.similarity_search_with_score(
        query,
        k=vectorstore.index.ntotal
    )

    # Standardize furnishing preferences.
    furnishing_aliases = {
        "FURNISHED": "YES",
        "FULLY FURNISHED": "YES",
        "UNFURNISHED": "NO",
        "NOT FURNISHED": "NO",
        "SEMI-FURNISHED": "PARTLY",
        "SEMI FURNISHED": "PARTLY",
        "PARTLY FURNISHED": "PARTLY",
        "PARTLY": "PARTLY",
        "YES": "YES",
        "NO": "NO",
        "UNKNOWN": "UNKNOWN"
    }

    required_furnishing = None
    if furnished is not None:
        requested = str(furnished).strip().upper()
        required_furnishing = furnishing_aliases.get(
            requested, requested
        )

    # Accept one amenity or a list of required amenities.
    if amenities is None:
        required_amenities = []
    elif isinstance(amenities, str):
        required_amenities = [amenities]
    else:
        required_amenities = list(amenities)

    matching_properties = []
    total_matches = 0

    for doc, distance in ranked_results:
        metadata = doc.metadata

        if metadata.get("source_type") != "property_listing":
            continue

        # Only monthly rental listings are eligible.
        if str(metadata.get("price_period", "")).lower() != "monthly":
            continue

        # Budget filter.
        price = pd.to_numeric(
            metadata.get("price_egp"),
            errors="coerce"
        )

        if max_budget is not None:
            if pd.isna(price) or float(price) > float(max_budget):
                continue

        # Bedroom filter, including Studio as a supported category.
        if bedrooms is not None:
            actual_bedrooms = str(
                metadata.get("bedrooms", "")
            ).strip().casefold()

            requested_bedrooms = str(bedrooms).strip().casefold()

            if actual_bedrooms != requested_bedrooms:
                continue

        # Location filter searches across available location fields.
        if location:
            location_text = " ".join(
                str(metadata.get(field, ""))
                for field in [
                    "city", "location_full", "town",
                    "district", "subdistrict"
                ]
            ).casefold()

            if str(location).strip().casefold() not in location_text:
                continue

        # Furnishing filter.
        if required_furnishing is not None:
            actual_furnishing = str(
                metadata.get("furnished", "UNKNOWN")
            ).strip().upper()

            if actual_furnishing != required_furnishing:
                continue

        # Every requested amenity must be present in the listing's
        # structured amenities field.
        available_amenities = str(
            metadata.get("amenities", "")
        ).casefold()

        if not all(
            str(item).strip().casefold() in available_amenities
            for item in required_amenities
        ):
            continue

        total_matches += 1

        if len(matching_properties) < top_k:
            matching_properties.append((doc, distance))

    return total_matches, matching_properties


# Test the reusable function with our original search.
query = (
    "I want to rent a two-bedroom apartment in New Cairo "
    "with monthly rent around 40000 EGP."
)

match_count, results = search_properties(
    query=query,
    max_budget=40000,
    bedrooms=2,
    location="New Cairo",
    top_k=5
)

print("=== Reusable Property Search Test ===")
print(f"Total matching properties: {match_count:,}")
print(f"Properties displayed: {len(results)}")

for rank, (doc, distance) in enumerate(results, start=1):
    metadata = doc.metadata

    print(f"\n--- Property {rank} ---")
    print(f"Location: {metadata.get('location_full')}")
    print(f"Monthly rent: {metadata.get('price_egp'):,.0f} EGP")
    print(f"Bedrooms: {metadata.get('bedrooms')}")
    print(f"Furnished: {metadata.get('furnished')}")
    print(f"Amenities: {metadata.get('amenities', 'Not specified')}")
    print(f"URL: {metadata.get('detail_url', '')}")
    print(f"Semantic distance: {distance:.4f}")

=== Reusable Property Search Test ===
Total matching properties: 799
Properties displayed: 5

--- Property 1 ---
Location: The Address East, 90 Street, The 5th Settlement, New Cairo City, Cairo
Monthly rent: 20,000 EGP
Bedrooms: 2
Furnished: PARTLY
Amenities: Balcony | Private Garden | Shared Spa | Shared Gym | Security | Lobby in Building
URL: https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-the-5th-settlement-90-street-the-address-east-8705718.html
Semantic distance: 0.4082

--- Property 2 ---
Location: The Village, South Investors Area, New Cairo City, Cairo
Monthly rent: 33,000 EGP
Bedrooms: 2
Furnished: PARTLY
Amenities: Balcony | Security
URL: https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-south-investors-area-the-village-2Ps4J7eplhI.html
Semantic distance: 0.4373

--- Property 3 ---
Location: The Village, South Investors Area, New Cairo City, Cairo
Monthly rent: 35,000 EGP
Bedrooms: 2
Furnished: PARTLY
Amenities

In [ ]:
# Test the additional furnishing and amenity filters

match_count, results = search_properties(
    query=(
        "I want a partly furnished two-bedroom apartment "
        "in New Cairo with a balcony for rent."
    ),
    max_budget=40000,
    bedrooms=2,
    location="New Cairo",
    furnished="PARTLY",
    amenities=["Balcony"],
    top_k=5
)

print("=== Furnishing and Amenity Filter Test ===")
print("Maximum monthly rent: 40,000 EGP")
print("Bedrooms: 2")
print("Location: New Cairo")
print("Furnishing: Partly furnished")
print("Required amenity: Balcony")
print(f"\nTotal matching properties: {match_count:,}")
print(f"Properties displayed: {len(results)}")

for rank, (doc, distance) in enumerate(results, start=1):
    metadata = doc.metadata

    print(f"\n--- Property {rank} ---")
    print(f"Location: {metadata.get('location_full')}")
    print(f"Monthly rent: {metadata.get('price_egp'):,.0f} EGP")
    print(f"Bedrooms: {metadata.get('bedrooms')}")
    print(f"Furnished: {metadata.get('furnished')}")
    print(f"Amenities: {metadata.get('amenities', '')}")
    print(f"URL: {metadata.get('detail_url', '')}")

=== Furnishing and Amenity Filter Test ===
Maximum monthly rent: 40,000 EGP
Bedrooms: 2
Location: New Cairo
Furnishing: Partly furnished
Required amenity: Balcony

Total matching properties: 217
Properties displayed: 5

--- Property 1 ---
Location: The Village, South Investors Area, New Cairo City, Cairo
Monthly rent: 35,000 EGP
Bedrooms: 2
Furnished: PARTLY
Amenities: Balcony | Built in Wardrobes | Central A/C | Covered Parking | Security
URL: https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-south-investors-area-the-village-2PtcVg506bY.html

--- Property 2 ---
Location: Moon Residences, Fifth Square, The 5th Settlement, New Cairo City, Cairo
Monthly rent: 35,000 EGP
Bedrooms: 2
Furnished: PARTLY
Amenities: Balcony | Built in Wardrobes | Central A/C | Covered Parking | Security
URL: https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-the-5th-settlement-fifth-square-moon-residences-2PtSJEcX3J2.html

--- Property 3 ---
Locati

In [ ]:
from langchain_community.vectorstores import FAISS

# Create a separate index containing only the housing-guide PDF chunks
guide_vectorstore = FAISS.from_documents(
    documents=pdf_chunks_filtered,
    embedding=embedding_model
)

# Retrieve guidance directly from the PDF index
def search_housing_guide(query, top_k=3):
    return guide_vectorstore.similarity_search_with_score(
        query,
        k=top_k
    )

print("=== Housing Guide Index ===")
print(f"PDF chunks indexed: {guide_vectorstore.index.ntotal}")
print("Housing guide retrieval function: Ready")

=== Housing Guide Index ===
PDF chunks indexed: 93
Housing guide retrieval function: Ready


In [ ]:
# Test the function with a practical rental question
guide_query = (
    "What should I check before renting an apartment in Egypt? "
    "Explain the rental agreement, apartment condition, bills, "
    "deposit, and additional costs."
)

guide_results = search_housing_guide(guide_query, top_k=3)

print("=== Housing Guide Retrieval Test ===")
print(f"Question: {guide_query}")
print(f"Guidance chunks retrieved: {len(guide_results)}")

for rank, (doc, distance) in enumerate(guide_results, start=1):
    print(f"\n--- Guidance Result {rank} ---")
    print(f"PDF page: {doc.metadata.get('page', 'Unknown')}")
    print(f"Semantic distance: {distance:.4f}")
    print(doc.page_content[:700].strip())
    print("-" * 50)

=== Housing Guide Retrieval Test ===
Question: What should I check before renting an apartment in Egypt? Explain the rental agreement, apartment condition, bills, deposit, and additional costs.
Guidance chunks retrieved: 3

--- Guidance Result 1 ---
PDF page: 18
Semantic distance: 0.5680
● Go over the points mentioned earlier. 
 
● Know ALL the additional costs: 
Regular bills (gas, water, electricity) and payments details. In addition to the usual costs, there are additional costs 
linked to renting in Egypt: 
- the bawwab (the doorman/superintendent). He’s paid 100 to 200 EGP a month per person; the price varies 
depending on the area and the type of building; 
- the zabbal (garbage collector) He’s paid 50 to 150 EGP each month per flat and the price varies depending on 
the area and the type of building. 
 
● Inspect the flat: 
It’s an important step because the landlords are often reluctant to have work done once the flat is occupied. If any 
work is necessary, ask that
-----------

In [ ]:
# Test focused searches for different rental-advice topics

focused_queries = {
    "Lease and deposit": (
        "rental lease agreement, contract terms, security deposit, "
        "and agreement between landlord and tenant"
    ),
    "Apartment condition and repairs": (
        "inspect the apartment before renting, existing damage, "
        "necessary repairs, and documenting apartment condition"
    ),
    "Bills and additional costs": (
        "gas, electricity and water bills, doorman fees, "
        "garbage collection fees, and other rental expenses"
    )
}

for topic, query in focused_queries.items():
    print(f"\n{'=' * 60}")
    print(f"Topic: {topic}")
    print(f"Search query: {query}")
    print("=" * 60)

    results = guide_vectorstore.similarity_search_with_score(
        query,
        k=2
    )

    for rank, (doc, distance) in enumerate(results, start=1):
        print(f"\nResult {rank}")
        print(f"PDF page: {doc.metadata.get('page', 'Unknown')}")
        print(f"Semantic distance: {distance:.4f}")
        print(doc.page_content[:450].strip())
        print("-" * 40)


Topic: Lease and deposit
Search query: rental lease agreement, contract terms, security deposit, and agreement between landlord and tenant

Result 1
PDF page: 19
Semantic distance: 1.0542
18 
 
2. As soon as the landlord leaves, change the locks. Some landlords enter the flats when tenants are not 
home. 

• 
The lease 
Only sign the lease if it’s written in English and mentions clearly the conditions for the return of the deposit and 
the advance notice of departure. 
Before you sign, check in particular that: 
- 
The return of the deposit is not linked to an absurd condition like requiring you to find another tenant 
when you le
----------------------------------------

Result 2
PDF page: 17
Semantic distance: 1.2304
favourable exchange rate. 
 
You don’t need to prepare files to give during the visits like you’d do in France for example, as only an ID is 
required when you sign the lease: the landlord uses it to declare their tenants. There are no proper real estate
---------------

In [ ]:
# Improve housing-guide retrieval for questions covering multiple topics

def search_housing_guide(query, top_k=3):
    query_lower = query.casefold()

    # Identify topics mentioned in the question
    topic_rules = [
        (
            "Lease and deposit",
            ["lease", "deposit", "contract", "landlord",
             "tenant", "sign", "agreement"],
            "rental lease agreement, contract terms, "
            "security deposit, and agreement between landlord and tenant"
        ),
        (
            "Apartment condition",
            ["inspect", "condition", "repair", "damage",
             "apartment", "flat", "move in"],
            "inspect the apartment before renting, existing damage, "
            "necessary repairs, and documenting apartment condition"
        ),
        (
            "Bills and additional costs",
            ["bill", "cost", "expense", "gas", "water",
             "electricity", "utility", "doorman", "garbage"],
            "gas, electricity and water bills, doorman fees, "
            "garbage collection fees, and other rental expenses"
        )
    ]

    active_topics = [
        (name, focused_query)
        for name, keywords, focused_query in topic_rules
        if any(keyword in query_lower for keyword in keywords)
    ]

    # Fall back to the original question if no topic is recognized
    if not active_topics:
        active_topics = [("General", query)]

    candidates = []
    seen = set()
    topic_best_results = []

    # Search each identified topic separately
    for topic, focused_query in active_topics:
        results = guide_vectorstore.similarity_search_with_score(
            focused_query,
            k=max(top_k, 2)
        )

        if results:
            topic_best_results.append(results[0])

        for doc, distance in results:
            key = (doc.metadata.get("page"), doc.page_content)

            if key not in seen:
                seen.add(key)
                candidates.append((doc, distance))

    # Prefer the best result from each topic to improve coverage
    selected = []
    selected_keys = set()

    for doc, distance in topic_best_results:
        key = (doc.metadata.get("page"), doc.page_content)

        if key not in selected_keys:
            selected.append((doc, distance))
            selected_keys.add(key)

        if len(selected) >= top_k:
            return selected

    # Fill any remaining places with the closest unused results
    candidates.sort(key=lambda item: item[1])

    for doc, distance in candidates:
        key = (doc.metadata.get("page"), doc.page_content)

        if key not in selected_keys:
            selected.append((doc, distance))
            selected_keys.add(key)

        if len(selected) >= top_k:
            break

    return selected


print("Topic-aware housing-guide retrieval: Ready")

Topic-aware housing-guide retrieval: Ready


# **LLM**

In [ ]:
!pip install -q langchain langchain-core langchain-google-genai pydantic

In [ ]:
from typing import Optional, Literal, List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

In [ ]:
api_key = userdata.get("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    google_api_key=api_key
)

# **Schema**

In [ ]:
class SearchIntent(BaseModel):
    intent: Literal["property_search", "housing_guide", "both", "chitchat"] = Field(
        description="property_search = looking for apartments; "
                    "housing_guide = advice about renting/settling in Egypt; "
                    "both = needs both; chitchat = greeting or unrelated"
    )
    max_budget: Optional[float] = Field(None, description="Max monthly rent in EGP")
    bedrooms: Optional[int] = Field(None, description="Number of bedrooms")
    location: Optional[str] = Field(None, description="Area/city, e.g. New Cairo")
    furnished: Optional[Literal["YES", "NO", "PARTLY"]] = Field(None)
    amenities: List[str] = Field(default_factory=list, description="e.g. Balcony, Parking")


class HousingAnswer(BaseModel):
    answer: str = Field(description="Answer to the user, based only on the context")
    insufficient_information: bool = Field(
        description="True if the context doesn't contain enough information"
    )

# **Router Chain**

In [ ]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You extract search intent and filters from a user's message about "
     "housing in Egypt. Only fill a filter if the user explicitly mentioned it. "
     "Never guess values."),
    ("human", "{question}")
])

router = router_prompt | llm.with_structured_output(SearchIntent)

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a housing assistant for students and newcomers in Egypt.\n"
     "Answer using ONLY the context. Do not invent prices, locations or details.\n"
     "If the context is insufficient, say so and set insufficient_information=true.\n"
     "Reply in the same language as the user's question.\n\n"
     "Context:\n{context}"),
    ("human", "{question}")
])

answer_chain = answer_prompt | llm.with_structured_output(HousingAnswer)

In [ ]:
def format_properties(results):
    blocks, sources = [], []
    for i, (doc, _) in enumerate(results, start=1):
        m = doc.metadata
        blocks.append(
            f"[Property {i}]\n"
            f"Location: {m.get('location_full', 'N/A')}\n"
            f"Monthly rent: {float(m.get('price_egp', 0)):,.0f} EGP\n"
            f"Bedrooms: {m.get('bedrooms', 'N/A')}\n"
            f"Furnished: {m.get('furnished', 'N/A')}\n"
            f"Amenities: {m.get('amenities', 'N/A')}\n"
            f"URL: {m.get('detail_url', '')}"
        )
        if m.get("detail_url"):
            sources.append(m["detail_url"])
    return "\n\n".join(blocks), sources


def format_guide(results):
    blocks, sources = [], []
    for i, (doc, _) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        blocks.append(f"[Guide {i} - page {page}]\n{doc.page_content}")
        sources.append(f"Housing Guide, page {page}")
    return "\n\n".join(blocks), sources

# **Pipeline**

In [ ]:
def ask(question: str) -> dict:

    intent = router.invoke({"question": question})

    if intent.intent == "chitchat":
        return {"answer": "أنا مساعد سكن، اسأليني عن شقق للإيجار أو نصايح الاستقرار في مصر.",
                "sources": [], "intent": intent}

    context_parts, sources = [], []


    if intent.intent in ("property_search", "both"):
        if not intent.location and not intent.max_budget:
            return {"answer": "ممكن تحدديلي المنطقة أو الميزانية عشان أدورلك صح؟",
                    "sources": [], "intent": intent}

        kwargs = dict(
            query=question,
            max_budget=intent.max_budget if intent.max_budget else float("inf"),
            location=intent.location or "",
            top_k=5,
        )
        if intent.bedrooms is not None:
            kwargs["bedrooms"] = intent.bedrooms
        if intent.furnished:
            kwargs["furnished"] = intent.furnished
        if intent.amenities:
            kwargs["amenities"] = intent.amenities

        match_count, prop_results = search_properties(**kwargs)

        if prop_results:
            text, src = format_properties(prop_results)
            context_parts.append(f"Matching properties ({match_count} total):\n{text}")
            sources += src
        else:
            context_parts.append("No properties matched the user's filters.")

    # 3) ابحث في الدليل
    if intent.intent in ("housing_guide", "both"):
        guide_results = search_housing_guide(question, top_k=3)
        if guide_results:
            text, src = format_guide(guide_results)
            context_parts.append(f"Housing guide excerpts:\n{text}")
            sources += src

    # 4) ولّد الإجابة
    context = "\n\n".join(context_parts)
    result = answer_chain.invoke({"question": question, "context": context})

    return {
        "answer": result.answer,
        "sources": [] if result.insufficient_information else list(dict.fromkeys(sources)),
        "intent": intent,
    }

# **Testing**

In [ ]:
tests = [
     "hello",                                              # المفروض يرد بترحيب
    "عايزة شقة",                                          # المفروض يسأل عن المنطقة أو الميزانية
    "apartment in New Cairo under 500 EGP",               # المفروض يقول مفيش نتايج
    "عايزة شقة غرفتين في التجمع بحد أقصى 30000",          # عربي: بيشتغل ولا لأ
    "How much is the metro ticket?",
]

for q in tests:
    out = ask(q)
    print("Q:", q)
    print("Intent:", out["intent"].model_dump())
    print("A:", out["answer"])
    print("Sources:", out["sources"])
    print("-" * 60)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: hello
Intent: {'intent': 'chitchat', 'max_budget': None, 'bedrooms': None, 'location': None, 'furnished': None, 'amenities': []}
A: أنا مساعد سكن، اسأليني عن شقق للإيجار أو نصايح الاستقرار في مصر.
Sources: []
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: عايزة شقة
Intent: {'intent': 'property_search', 'max_budget': None, 'bedrooms': None, 'location': None, 'furnished': None, 'amenities': []}
A: ممكن تحدديلي المنطقة أو الميزانية عشان أدورلك صح؟
Sources: []
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: apartment in New Cairo under 500 EGP
Intent: {'intent': 'property_search', 'max_budget': 500.0, 'bedrooms': None, 'location': 'New Cairo', 'furnished': None, 'amenities': []}
A: No properties matched the user's filters.
Sources: []
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: عايزة شقة غرفتين في التجمع بحد أقصى 30000
Intent: {'intent': 'property_search', 'max_budget': 30000.0, 'bedrooms': 2, 'location': 'التجمع', 'furnished': None, 'amenities': []}
A: عذراً، لا توجد عقارات مطابقة للفلاتر الخاصة بك.
Sources: []
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: How much is the metro ticket?
Intent: {'intent': 'chitchat', 'max_budget': None, 'bedrooms': None, 'location': None, 'furnished': None, 'amenities': []}
A: أنا مساعد سكن، اسأليني عن شقق للإيجار أو نصايح الاستقرار في مصر.
Sources: []
------------------------------------------------------------


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a housing assistant for students and newcomers in Egypt.\n"
     "Facts about apartments and housing advice must come ONLY from the Context. "
     "Do not invent prices, locations or details.\n"
     "You MAY use the Conversation history to understand follow-up questions and to "
     "answer questions about the conversation itself (for example what the user asked "
     "earlier, or which of the properties already shown is the cheapest).\n"
     "For greetings or small talk, reply briefly and politely and offer to help with "
     "apartments or housing advice.\n"
     "If the Context and the history are not enough, say so and set "
     "insufficient_information=true.\n"
     "Reply in the same language as the user's question.\n\n"
     "Conversation history:\n{chat_history}\n\n"
     "Context:\n{context}"),
    ("human", "{question}")
])

answer_chain = answer_prompt | llm.with_structured_output(HousingAnswer)


In [ ]:
def ask(question: str, chat_history: str = "(no previous messages)") -> dict:  # <-- NEW (chat_history)
    if isinstance(question, (list, tuple)):
        question = " ".join(map(str, question))
    question = str(question).strip()

    intent = router.invoke({"question": question})
    context_parts, sources = [], []

    # 1) شقق
    if intent.intent in ("property_search", "both"):
        if not intent.location and not intent.max_budget:
            return {"answer": "ممكن تحدديلي المنطقة أو الميزانية عشان أدورلك صح؟",
                    "sources": [], "intent": intent}

        kwargs = dict(
            query=question,
            max_budget=intent.max_budget if intent.max_budget else float("inf"),
            location=intent.location or "",
            top_k=5,
        )
        if intent.bedrooms is not None:
            kwargs["bedrooms"] = intent.bedrooms
        if intent.furnished:
            kwargs["furnished"] = intent.furnished
        if intent.amenities:
            kwargs["amenities"] = intent.amenities

        match_count, prop_results = search_properties(**kwargs)

        if prop_results:
            text, src = format_properties(prop_results)
            context_parts.append(f"Matching properties ({match_count} total):\n{text}")
            sources += src
        else:
            context_parts.append("No properties matched the user's filters.")

    # 2) دليل السكن
    if intent.intent in ("housing_guide", "both"):
        guide_results = search_housing_guide(question, top_k=3)
        if guide_results:
            text, src = format_guide(guide_results)
            context_parts.append(f"Housing guide excerpts:\n{text}")
            sources += src

    # <-- NEW: الـ chitchat والأسئلة عن المحادثة بتعدّي على الـ LLM بدل رد جاهز
    if not context_parts:
        context_parts.append("No retrieval was needed for this message.")

    result = answer_chain.invoke({
        "question": question,
        "context": "\n\n".join(context_parts),
        "chat_history": chat_history,                      # <-- NEW
    })

    return {
        "answer": result.answer,
        "sources": [] if result.insufficient_information else list(dict.fromkeys(sources)),
        "intent": intent,
    }


In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You maintain a running summary of a conversation between a user and a housing "
     "assistant for Egypt. Merge the new lines into the existing summary. Keep the "
     "user's requirements and preferences (area, budget, bedrooms, furnishing, "
     "amenities), the properties or advice already presented, and any unanswered "
     "question. Be concise (max 150 words). Return ONLY the updated summary."),
    ("human", "Existing summary:\n{summary}\n\nNew lines:\n{new_lines}")
])
summary_chain = summary_prompt | llm


def _text(content) -> str:
    """Gemini ساعات بيرجّع content كـ list من blocks، فنحوّله لنص."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
        return "".join(parts)
    return str(content)


class ChatMemory:
    """
    use_summary=False -> Buffer memory: كل المحادثة بتتبعت للـ LLM.
    use_summary=True  -> Buffer Summary memory: آخر keep_recent تبادلات حرفياً
                         + ملخص للتبادلات الأقدم (بيتحدّث تلقائياً).
    full_history دايماً بيحتفظ بالمحادثة كاملة حتى في وضع الملخص.
    """

    def __init__(self, use_summary: bool = False, keep_recent: int = 4):
        self.full_history = []          # [(question, answer), ...] مش بتتقصّ أبداً
        self.use_summary = use_summary
        self.keep_recent = keep_recent
        self.summary = ""
        self._summarized_upto = 0       # عدد التبادلات اللي اتلخّصت

    def add(self, question: str, answer: str):
        self.full_history.append((question, answer))
        if self.use_summary:
            pending = len(self.full_history) - self._summarized_upto
            if pending > self.keep_recent:
                self._summarize()

    def _summarize(self):
        end = len(self.full_history) - self.keep_recent
        old = self.full_history[self._summarized_upto:end]
        new_lines = "\n".join(f"User: {q}\nAssistant: {a[:600]}" for q, a in old)
        self.summary = _text(
            summary_chain.invoke(
                {"summary": self.summary or "(empty)", "new_lines": new_lines}
            ).content
        ).strip()
        self._summarized_upto = end

    def history_text(self) -> str:
        """النص اللي بيتبعت للـ LLM (ملخص + آخر التبادلات، أو المحادثة كلها)."""
        parts = []
        if self.summary:
            parts.append("Summary of the earlier conversation:\n" + self.summary)
        start = self._summarized_upto if self.use_summary else 0
        for q, a in self.full_history[start:]:
            parts.append(f"User: {q}\nAssistant: {a}")
        return "\n\n".join(parts) if parts else "(no previous messages)"

    def export(self):
        return [{"question": q, "answer": a} for q, a in self.full_history]

    def clear(self):
        self.__init__(self.use_summary, self.keep_recent)



In [ ]:
condense_prompt = ChatPromptTemplate.from_messages([                   # بيستخدم شات هيستوري عشان يحول الأسئلة اللي بتعتمد ع محادثة سابقة لأسئلة مفهومة
    ("system",
     "Rewrite the user's latest message as ONE standalone question, using the chat "
     "history for any missing details (area, budget, bedrooms, furnishing, amenities, "
     "or which property they mean).\n"
     "- If the message is already standalone, return it unchanged.\n"
     "- If it changes one detail (e.g. 'what about cheaper ones?'), keep the other "
     "details from the history.\n"
     "- If it is about the conversation itself (e.g. 'what did I ask before?') or is "
     "a greeting, return it unchanged.\n"
     "- If it starts a completely new topic, do NOT carry over old details.\n"
     "- Keep the user's language. Return ONLY the question."),
    ("human", "Chat history:\n{history}\n\nLatest message: {question}")
])
condense_chain = condense_prompt | llm


def chat(question: str, memory: ChatMemory) -> dict:
    question = str(question).strip()
    history = memory.history_text()

    if memory.full_history:
        standalone = _text(
            condense_chain.invoke({"history": history, "question": question}).content
        ).strip() or question
    else:
        standalone = question

    out = ask(standalone, chat_history=history)
    memory.add(question, out["answer"])
    out["standalone_question"] = standalone
    return out


In [ ]:
import re, time
                                                            #تتعامل مع مشكلة تجاوز ال API
def safe_chat(question, memory, retries=4):
    """بتعيد المحاولة تلقائياً لو Gemini رجّع 429 (تعدّي حد الاستخدام)."""
    for attempt in range(retries):
        try:
            return chat(question, memory)
        except Exception as e:
            err = str(e)
            if "429" in err or "RESOURCE_EXHAUSTED" in err:
                m = re.search(r"retry in ([\d.]+)s", err)
                wait = min(float(m.group(1)) + 2 if m else 25, 65)
                print(f"⏳ تعدّينا حد الاستخدام، هستنى {wait:.0f} ثانية...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("الحد لسه متعدّي، جرّبي بعد دقيقة.")


memory = ChatMemory(use_summary=False)

for q in [
    "I want a 2-bedroom apartment in New Cairo under 40000 EGP",
    "What about partly furnished ones?",
    "Which one has a balcony?",
    "What should I check before signing the lease?",   # موضوع جديد
    "What was my first question?",                      # سؤال عن المحادثة نفسها
]:
    out = safe_chat(q, memory)
    print("User:", q)
    print("Standalone:", out["standalone_question"])
    print("A:", out["answer"][:400])
    print("-" * 60)

print("Stored turns:", len(memory.full_history))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


User: I want a 2-bedroom apartment in New Cairo under 40000 EGP
Standalone: I want a 2-bedroom apartment in New Cairo under 40000 EGP
A: I found several 2-bedroom apartments in New Cairo under 40,000 EGP: 1) The Address East, The 5th Settlement for 20,000 EGP per month. 2) The 1st Settlement for 18,000 EGP per month. 3) Al Narges 4 for 25,000 EGP per month. 4) Fifth Square, The 5th Settlement for 23,000 EGP per month. 5) Mountain View iCity, The 5th Settlement for 35,000 EGP per month.
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


User: What about partly furnished ones?
Standalone: What about partly furnished 2-bedroom apartments in New Cairo under 40000 EGP?
A: All the 2-bedroom apartments in New Cairo under 40,000 EGP shown previously are partly furnished: 1) The Address East, The 5th Settlement for 20,000 EGP per month. 2) The Village, South Investors Area for 33,000 EGP per month. 3) Moon Residences, Fifth Square, The 5th Settlement for 35,000 EGP per month. 4) The Village, South Investors Area for 35,000 EGP per month. 5) Al Narges 4 for 25,000 EGP p
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


User: Which one has a balcony?
Standalone: Which of the 2-bedroom apartments in New Cairo under 40000 EGP that are partly furnished has a balcony?
A: All of the partly furnished 2-bedroom apartments in New Cairo under 40,000 EGP from the current list have a balcony: 1) The Address East, The 5th Settlement for 20,000 EGP per month. 2) Hyde Park for 40,000 EGP per month. 3) The Village, South Investors Area for 35,000 EGP per month. 4) Galleria Residences, South Investors Area for 30,000 EGP per month. 5) The Village, South Investors Area for 35,
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


User: What should I check before signing the lease?
Standalone: What should I check before signing the lease?
A: Before signing the lease, you should check in particular that: 1) The return of the deposit is not linked to an absurd condition like requiring you to find another tenant when you leave. 2) The end date on the lease matches your situation. 3) The conditions for the departure and in particular for the advance notice satisfy you. Additionally, ensure the lease is written in English and clearly menti
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


User: What was my first question?
Standalone: What was my first question?
A: Your first question was: "I want a 2-bedroom apartment in New Cairo under 40000 EGP"
------------------------------------------------------------
Stored turns: 5


In [ ]:
out = safe_chat("I want to rent an apartment having two bedrooms with maximum budget of 30000EGP in New Cairo", memory)
print(out["answer"])
print(out["sources"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I found several 2-bedroom apartments in New Cairo with a maximum budget of 30,000 EGP: 1) The Village, South Investors Area for 30,000 EGP per month (partly furnished). 2) District 5, 5th Settlement for 30,000 EGP per month (unfurnished). 3) Cairo Festival City, North Investors Area for 1,750 EGP per month (fully furnished). 4) Galleria Residences, South Investors Area for 30,000 EGP per month (partly furnished). 5) Mountain View iCity, The 5th Settlement for 30,000 EGP per month (fully furnished).
['https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-south-investors-area-the-village-2PcsGzG9ufQ.html', 'https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-the-5th-settlement-5th-settlement-compounds-district-5-2PWUofz7moi.html', 'https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-north-investors-area-cairo-festival-city-2PYhaQjxJZ2.html', 'https://www.propertyfinder.eg/en/plp/rent/apartment-for-ren

In [ ]:
out = safe_chat("", memory)

print("Standalone:", out["standalone_question"])
print("Intent:", out["intent"].model_dump())
print("A:", out["answer"])
print("Sources:", out["sources"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Standalone: I want to rent an apartment having two bedrooms with maximum budget of 30000EGP in New Cairo
Intent: {'intent': 'property_search', 'max_budget': 30000.0, 'bedrooms': 2, 'location': 'New Cairo', 'furnished': None, 'amenities': []}
A: I found several 2-bedroom apartments in New Cairo with a maximum budget of 30,000 EGP: 1) The Village, South Investors Area for 30,000 EGP per month (partly furnished). 2) District 5, 5th Settlement for 30,000 EGP per month (unfurnished). 3) Cairo Festival City, North Investors Area for 1,750 EGP per month (fully furnished). 4) Galleria Residences, South Investors Area for 30,000 EGP per month (partly furnished). 5) Mountain View iCity, The 5th Settlement for 30,000 EGP per month (fully furnished).
Sources: ['https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-south-investors-area-the-village-2PcsGzG9ufQ.html', 'https://www.propertyfinder.eg/en/plp/rent/apartment-for-rent-cairo-new-cairo-city-the-5th-settlement-5th-